# HW2 — Part 2: Warehouse Robot with PPO
### Custom Gymnasium Environment · Stable-Baselines3 PPO
**Yogeshvar Reddy Kallam** · IST 597 Deep RL · Penn State Spring 2025

---

## Problem 2: Proximal Policy Optimization — Warehouse Robot

**Environment:** 1-D corridor (7 positions). Robot picks up packages at centre (pos=3) and delivers to the correct endpoint.

```
Positions: [0]─[1]─[2]─[3]─[4]─[5]─[6]
                      ↑ pick-up
         drop-left            drop-right
```

**State:** (position, package_state)  
**Actions:** MOVE_LEFT | MOVE_RIGHT | PICK_UP | DROP_OFF  
**Algorithm:** PPO — clipped surrogate objective, MLP policy

In [ ]:
import random
import numpy as np
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.evaluation import evaluate_policy

NO_PACKAGE=0; PACKAGE_LEFT=1; PACKAGE_RIGHT=2
MOVE_LEFT=0; MOVE_RIGHT=1; PICK_UP=2; DROP_OFF=3

class WarehouseRobotEnv(gym.Env):
    def __init__(self, max_steps=200):
        super().__init__()
        self.observation_space = spaces.MultiDiscrete([7, 3])
        self.action_space      = spaces.Discrete(4)
        self.max_steps = max_steps; self.reset()

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.position=3; self.package=NO_PACKAGE; self.step_count=0
        return (self.position, self.package), {}

    def step(self, action):
        r = 0
        if action==MOVE_LEFT  and random.random()<0.95: self.position=max(self.position-1,0)
        elif action==MOVE_RIGHT and random.random()<0.95: self.position=min(self.position+1,6)
        elif action==PICK_UP and self.package==NO_PACKAGE and self.position==3:
            self.package=random.choice([PACKAGE_LEFT,PACKAGE_RIGHT])
        elif action==DROP_OFF and self.package!=NO_PACKAGE:
            r = 1 if ((self.package==PACKAGE_LEFT and self.position==0) or
                      (self.package==PACKAGE_RIGHT and self.position==6)) else -0.1
            self.package=NO_PACKAGE
        self.step_count+=1
        return (self.position,self.package), r, False, self.step_count>=self.max_steps, {}

env = WarehouseRobotEnv()
vec = DummyVecEnv([lambda: WarehouseRobotEnv()])
model = PPO("MlpPolicy", vec, verbose=0, gamma=0.99, learning_rate=3e-4)
print("Training PPO (100k steps)...")
model.learn(total_timesteps=100_000, progress_bar=False)
mean_r, std_r = evaluate_policy(model, vec, n_eval_episodes=20, deterministic=True)
print(f"Mean reward: {mean_r:.2f} ± {std_r:.2f}")
